In [78]:
import pandas as pd

titles = [
    "Oppenheimer",
    "Barbie",
    "Dune: Part Two",
    "Madame Web",
    "Inside Out 2"
]

movies = pd.DataFrame({"title": titles})
display(movies)

,title
0,Oppenheimer
1,Barbie
2,Dune: Part Two
3,Madame Web
4,Inside Out 2


In [ ]:
import requests

OMDB_KEY = "690401ba"

def get_movie(title):
    response = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": OMDB_KEY, "t": title},
    )
    data = response.json()

    rt = next(
        (r["Value"] for r in data.get("Ratings", [])
         if r["Source"] == "Rotten Tomatoes"),
        None
    )

    return {
        "title": data["Title"],
        "imdb_id": data["imdbID"],
        "release_date": data.get("Released"),
        "genre": data.get("Genre"),
        "rt_score": int(rt.rstrip("%")) if rt else None
    }

records = [get_movie(title) for title in titles]

movies = pd.DataFrame(
    [record for record in records if record is not None]
)

movies["release_date"] = pd.to_datetime(
    movies["release_date"],
)

movies.to_csv("data/movies.csv", index=False)

display(movies)

,title,imdb_id,release_date,genre,rt_score
0,Oppenheimer,tt15398776,2023-07-21,"Biography, Drama, History",93
1,Barbie,tt1517268,2023-07-21,"Adventure, Comedy, Fantasy",88
2,Dune: Part Two,tt15239678,2024-03-01,"Action, Adventure, Drama",92
3,Madame Web,tt11057302,2024-02-14,"Action, Thriller",10
4,Inside Out 2,tt22022452,2024-06-14,"Animation, Adventure, Comedy",91


In [92]:
from datetime import timedelta
import time

ARCTIC_URL = (
    "https://arctic-shift.photon-reddit.com/api/posts/search"
)

def fetch_posts(title, release_date, subreddit):
    release = pd.Timestamp(release_date)

    end = release - pd.Timedelta(days=3)
    start = end - pd.Timedelta(days=30)

    params = {
        "subreddit": subreddit,
        "title": title,
        "after": start.strftime("%Y-%m-%d"),
        "before": end.strftime("%Y-%m-%d"),
        "limit": 25,
        "sort": "asc",
        "fields": (
            "id,title,subreddit,score,"
            "num_comments,created_utc,selftext"
        )
    }

    response = requests.get(ARCTIC_URL,params=params, timeout=60)
    response.raise_for_status()

    return response.json().get("data", [])  

In [93]:
all_posts = []
failures = []

movies = pd.read_csv("data/movies.csv", parse_dates=["release_date"])

for _, movie in movies.iterrows():
    for subreddit in ["movies", "boxoffice"]:
        try:
            results = fetch_posts(
                movie["title"],
                movie["release_date"],
                subreddit
            )

            for post in results:
                post["movie_id"] = movie["imdb_id"]
                post["movie_title"] = movie["title"]

            all_posts.extend(results)

        except requests.RequestException as e:
            failures.append({
                "movie": movie["title"],
                "subreddit": subreddit,
                "error": str(e)
            })

        time.sleep(1)

posts_df = pd.DataFrame(all_posts)

print("Posts collected:", len(posts_df))
print("Failed searches:", len(failures))

display(posts_df.head())
display(pd.DataFrame(failures))

Posts collected: 36
Failed searches: 8


,created_utc,id,num_comments,score,selftext,subreddit,title,movie_id,movie_title
0,1705593713,199tg6r,19,1,,movies,Character Posters for 'Madame Web',tt11057302,Madame Web
1,1705615268,19a273t,0,1,[deleted],movies,MADAME WEB – Who Is Madame Web | In Theaters F...,tt11057302,Madame Web
2,1705698854,19atyeu,3,1,,movies,Dakota Johnson discovered she’s ‘really good’ ...,tt11057302,Madame Web
3,1705955488,19d5i2g,1,1,,movies,Madame Web Tracking for Worse Box Office Openi...,tt11057302,Madame Web
4,1706189273,19f9bjh,2,1,,movies,MADAME WEB – Grounded In Reality,tt11057302,Madame Web


,movie,subreddit,error
0,Oppenheimer,movies,503 Server Error: Service Unavailable for url:...
1,Oppenheimer,boxoffice,503 Server Error: Service Unavailable for url:...
2,Barbie,movies,503 Server Error: Service Unavailable for url:...
3,Barbie,boxoffice,503 Server Error: Service Unavailable for url:...
4,Dune: Part Two,movies,503 Server Error: Service Unavailable for url:...
5,Dune: Part Two,boxoffice,503 Server Error: Service Unavailable for url:...
6,Madame Web,boxoffice,503 Server Error: Service Unavailable for url:...
7,Inside Out 2,movies,503 Server Error: Service Unavailable for url:...


In [85]:
posts_df = posts_df.drop_duplicates(subset=["movie_id", "id"])
posts_df["score"] = pd.to_numeric(posts_df["score"])
posts_df["num_comments"] = pd.to_numeric(posts_df["num_comments"],)
posts_df["created_utc"] = pd.to_datetime(posts_df["created_utc"], unit="s", utc=True,)
posts_df = posts_df.dropna(subset=["movie_id", "id", "created_utc"])

posts_df.to_csv("data/reddit_posts.csv", index=False)

KeyError: 'score'

In [ ]:
movie_stats = (
    posts_df.groupby("movie_id")
    .agg(
        total_posts=("id", "count"),
        total_comments=("num_comments", "sum"),
        avg_post_score=("score", "mean"),
        median_post_score=("score", "median"),
        max_post_score=("score", "max"),
        subreddit_count=("subreddit", "nunique")
    )
    .reset_index()
)

movie_features = movies.merge(
    movie_stats,
    left_on="imdb_id",
    right_on="movie_id",
    how="left"
)

movie_features["total_posts"] = (movie_features["total_posts"].fillna(0))
movie_features.to_csv("data/movie_features.csv",)

display(movie_features.head())

,title,imdb_id,release_date,genre,rt_score,movie_id,total_posts,total_comments,avg_post_score,median_post_score,max_post_score,subreddit_count
0,Oppenheimer,tt15398776,2023-07-21,"Biography, Drama, History",93,tt15398776,50.0,1797.0,127.400000,6.0,3679.0,2.0
1,Barbie,tt1517268,2023-07-21,"Adventure, Comedy, Fantasy",88,tt1517268,25.0,1693.0,93.800000,18.0,907.0,1.0
2,Dune: Part Two,tt15239678,2024-03-01,"Action, Adventure, Drama",92,tt15239678,19.0,1492.0,191.736842,16.0,1058.0,1.0
3,Madame Web,tt11057302,2024-02-14,"Action, Thriller",10,tt11057302,11.0,118.0,1.181818,1.0,3.0,1.0
4,Inside Out 2,tt22022452,2024-06-14,"Animation, Adventure, Comedy",91,NaN,0.0,NaN,NaN,NaN,NaN,NaN
